# Conservative RAD on convex polyhedra

Original polygonal macrofaces carry one P0 or one affine P1 flux polynomial. Tetrahedral local refinement does not add global traces on triangulation diagonals. The campaign uses the three-dimensional smooth PDE and P4/P1 degrees of DOI [10.1016/j.cma.2024.117089](https://doi.org/10.1016/j.cma.2024.117089), on explicitly constructed convex meshes.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.polyhedral import PolyhedralMesh
from pymhm.polyhedral_rad import solve_polyhedral_rad

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
mesh = PolyhedralMesh.cubes()
exact = lambda x: 1 + x[:, 0] + 2*x[:, 1] - x[:, 2]
patch = solve_polyhedral_rad(mesh, degree=3, dirichlet=exact)
assert patch.skeleton.size == 3*len(mesh.faces)
assert patch.l2_error(exact) < 1e-11
print("Polygonal face DOFs:", patch.skeleton.size, "affine L2 error:", patch.l2_error(exact))

## Analytical data and discrete spaces

On the unit cube, $u=\sin(6\pi x)\sin(4\pi y)\sin(2\pi z)$, $K=0.1I$, $\beta=(1,0,0)$ and $c=0$. Hence $f=5.6\pi^2u+\partial_xu$. The local natural multiplier is $(-K\nabla u+\beta u/2)\cdot n$; the reported physical flux is $-K\nabla u+\beta u$.

The classical exact solution supplies the independent error. The five macro resolutions and mesh geometries are original; these records do not claim possession of the historical article meshes.

In [ ]:
record = json.loads((root / "examples/results/polyhedral-rad.json").read_text())
rows = record["convergence"]
assert len(rows) == 15
for family in ("cube", "triangular-prism", "hexagonal-prism"):
    selected = sorted((row for row in rows if row["family"] == family), key=lambda row: row["n"])
    assert [row["n"] for row in selected] == [2, 3, 4, 5, 6]
    assert all(row["backward_residual"] < 1e-8 for row in selected)
    assert all(not row["source_changed"] for row in selected)
    for row in selected:
        print(f"{family:18s} n={row['n']}  L2={row['l2_error']:.6e}  V={row['V_error']:.6e}  flux={row['physical_flux_l2_error']:.6e}")

In [ ]:
for name in ("geometry", "convergence", "fields"):
    display(Image(filename=str(root / f"docs/figures/polyhedral-rad/{name}.png")))

The field panels are direct local P4 evaluations on the physical slice $z=0.37$. Outlines mark actual macroface intersections; broken values are not averaged. Error norms use positive volume quadrature rather than these image samples. The initial coarse scalar errors need not be monotone for this oscillatory problem. Geometry accepts convex cells with planar strictly convex faces; it does not support curved or nonconvex polyhedra.